In [ ]:
# Joahannes B D da Costa <joahannes.costa@unifesp.br>

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from matplotlib.ticker import MultipleLocator

In [ ]:
# cria o diretório para salvar os gráficos, se não existir
dir = "otimizacao"
if not os.path.isdir(dir):
    os.makedirs(dir, exist_ok=True)
    print(f"O diretório '{dir}' foi criado para salvar os gráficos.")
else:
    print(f"O diretório '{dir}' já existe. Os gráficos serão salvos nele.")

In [ ]:
path = '../system/output/'

output_path = "otimizacao/"

output_name = "Otimizacao"

ALGORITHM   = ['ORION', 'NSGA3', 'PSO']

LABELS = {
    'ORION' : 'NSGA-II',
    'NSGA3' : 'NSGA-III',
	'PSO'   : 'PSO'	
}

HATCHES = {
    'ORION' : '',
    'NSGA3' : 'o',
	'PSO'   : 'x'	
}

WITH_HATCHES = True

#VARIABLES PLOT
Y_LEGEND_SCHEDULED  = u'Tarefas Escalonadas (%)'
Y_LEGEND_LATENCY    = u'Latência do Sistema (s)'
Y_LEGEND_COST       = u'Custo Monetário ($)'
Y_LEGEND_CPU_TIME   = u'Tempo de CPU (ms)'

X_LEGEND            = u'Prazo (s)'

header = [
    "task_total","task_id", "task_size", "task_value", "task_cpu", "task_deadline", "task_insert_time", "task_start_time", "task_finish_time", "task_remove_time", "task_waiting_time", "task_cost", "task_status",
    "config_seed", "config_rate", "config_deadline", "algorithm"
    ]

In [ ]:
RATE        = [30,]  # [1, 2, 3, 4, 5, 10, 15]
CYCLES      = [30,] # [30,]
DEADLINES   = [0.5, 7] # [0.3, 0.8, 1, 3, 5, 7]
SEEDS       = [1, 2, 3, 4, 5] # [1, 2, 3, 4, 5]

CPU_CYCLE   = 30

In [ ]:
grid_config_lw    = 1.6
grid_config_alpha = 0.05
config_legend = (0.38, 1.15)

In [ ]:
df = pd.DataFrame(columns=header)

dataframes = []

for algorithm in ALGORITHM:

    for task_rate in RATE:

        for deadline in DEADLINES:

            for seed in SEEDS:
    
                arquivo = path + str(algorithm) + '/SEED_' + str(seed) + '_RESULTS_radius_2000_resource_1_weight_10_rate_' + str(task_rate) + '_megacycles_' + str(CPU_CYCLE) + '_deadline_' + str(deadline) + '.txt'
                df = pd.read_csv(arquivo, sep='\t', names=header)
                df['config_seed'] = seed
                df['config_rate'] = task_rate
                df['config_deadline'] = deadline
                df['algorithm'] = algorithm
                dataframes.append(df)

                # print(arquivo)

df_final = pd.concat(dataframes, ignore_index=True)

## Criando coluna para Latency

In [ ]:
df_final['result_latency'] = df_final['task_remove_time'] - df_final['task_insert_time']
df_final

## Configuração dos plots

In [ ]:
formato = ".png"

legend_size = 25

x_fig = 6.0 #6.8
y_fig = 7.5 #5.5

colors = ['#E6550D', '#1F77B4', '#7F7F7F']

color_style = colors
# color_style = "Reds"

font_size_algoritm = 23
legend_position = (1, 0.6, 0.03, 0.97)

YLIM_SCHEDULED  = (-1.5, 105)
YLIM_COST       = (-1.5, 150)
YLIM_LATENCY    = (-0.1, 6.2)
YLIM_CPU_TIME   = (-4.0, 270)

YTICKS_SCHEDULED  = 20
YTICKS_COST       = 20
YTICKS_LATENCY    = 1
YTICKS_CPU_TIME   = 50

# Scheduled tasks

In [ ]:
def plot_scheduled(TASK_RATE, ALGORITHM_LIST):

    df_scheduled = df_final[df_final['config_rate'] == TASK_RATE]

    sns.set(style="ticks")
    sns.set_palette(color_style)
    
    ax = plt.subplot()

    ax = sns.barplot(x='config_deadline', y=(df_scheduled['task_status'] == 'COMPLETED') * 100, data=df_scheduled, hue='algorithm', edgecolor='k', estimator=np.mean, errorbar=('ci', 95), capsize=.08, err_kws={'linewidth': 1})

    plt.xlabel(X_LEGEND, fontsize=legend_size)
    plt.ylabel(Y_LEGEND_SCHEDULED, fontsize=legend_size)

    xticks = np.arange(0,len(DEADLINES),1)

    plt.xticks(xticks, DEADLINES, fontsize=legend_size)
    plt.yticks(fontsize=legend_size)

    plt.ylim(YLIM_SCHEDULED)

    # ax.set_yticks(np.arange(0,110,10), labels=np.arange(0,110,10))
    # Define os intervalos dos ticks do eixo y para cada subplot
    ax.yaxis.set_major_locator(MultipleLocator(YTICKS_SCHEDULED))

    if WITH_HATCHES == True:
        hatches = HATCHES.values()
        # Loop over the bars
        for bars, hatch in zip(ax.containers, hatches):
            # Set a different hatch for each group of bars
            for bar in bars:
                bar.set_hatch(hatch)

    # Adiciona um bloco de legenda no topo dos subplots com hatches
    legend_handles, names = ax.get_legend_handles_labels()
    final_labels = []
    if WITH_HATCHES == True:
        for alg, handle, hatch in zip(names, legend_handles, HATCHES):
            print(alg, handle, hatch)
            final_name = LABELS[alg]
            final_labels.append(final_name)
            handle.set_hatch(HATCHES[alg])
    else:
        for alg, handle in zip(names, legend_handles):
            final_name = LABELS[alg]
            final_labels.append(final_name)

    ax.legend(
        legend_handles,
        final_labels,
        loc             = 'upper center',
        bbox_to_anchor  = config_legend,
        handletextpad   = 0.2,
        handlelength    = 2.2,
        handleheight    = 1.2,
        columnspacing   = 0.5,
        fancybox        = False,
        frameon         = False,
        ncol            = len(ALGORITHM),
        edgecolor       = 'k',
        fontsize        = font_size_algoritm
    )

    # adiciona grid ao fundo do plot
    plt.grid(color='k', linestyle='--', linewidth=grid_config_lw, axis='both', alpha=grid_config_alpha)

    fig = plt.gcf()
    fig.set_size_inches(x_fig, y_fig)
    fig.savefig(output_path + 'Escalonadas_' + str(TASK_RATE) + '_' + output_name + formato, dpi=200, bbox_inches = 'tight', pad_inches = 0.05)
    # plt.close()

    plt.show()


# Moneraty costs

In [ ]:
def plot_cost(TASK_RATE, ALGORITHM_LIST):

    df_cost = df_final[df_final['config_rate'] == TASK_RATE]

    sns.set(style="ticks")
    sns.set_palette(color_style)

    ax = plt.subplot()

    ax = sns.barplot(x='config_deadline', y='task_cost', data=df_cost, hue='algorithm', edgecolor='k', estimator=np.mean, errorbar=('ci', 95), capsize=.08, err_kws={'linewidth': 1})

    plt.xlabel(X_LEGEND, fontsize=legend_size)
    plt.ylabel(Y_LEGEND_COST, fontsize=legend_size)

    xticks = np.arange(0,len(DEADLINES),1)

    plt.xticks(xticks, DEADLINES, fontsize=legend_size)
    plt.yticks(fontsize=legend_size)

    # Define os intervalos dos ticks do eixo y para cada subplot
    ax.yaxis.set_major_locator(MultipleLocator(YTICKS_COST))

    plt.ylim(YLIM_COST)

    if WITH_HATCHES == True:
        hatches = HATCHES.values()
        # Loop over the bars
        for bars, hatch in zip(ax.containers, hatches):
            # Set a different hatch for each group of bars
            for bar in bars:
                bar.set_hatch(hatch)

    # Adiciona um bloco de legenda no topo dos subplots com hatches
    legend_handles, names = ax.get_legend_handles_labels()
    final_labels = []
    if WITH_HATCHES == True:
        for alg, handle, hatch in zip(names, legend_handles, HATCHES):
            print(alg, handle, hatch)
            final_name = LABELS[alg]
            final_labels.append(final_name)
            handle.set_hatch(HATCHES[alg])
    else:
        for alg, handle in zip(names, legend_handles):
            final_name = LABELS[alg]
            final_labels.append(final_name)

    ax.legend(
        legend_handles,
        final_labels,
        loc             = 'upper center',
        bbox_to_anchor  = config_legend,
        handletextpad   = 0.2,
        handlelength    = 2.2,
        handleheight    = 1.2,
        columnspacing   = 0.5,
        fancybox        = False,
        frameon         = False,
        ncol            = len(ALGORITHM),
        edgecolor       = 'k',
        fontsize        = font_size_algoritm
    )

    # adiciona grid ao fundo do plot
    plt.grid(color='k', linestyle='--', linewidth=grid_config_lw, axis='both', alpha=grid_config_alpha)

    fig = plt.gcf()
    fig.set_size_inches(x_fig, y_fig)
    fig.savefig(output_path + 'Custo_' + str(TASK_RATE) + '_' + output_name + formato, dpi=200, bbox_inches = 'tight', pad_inches = 0.05)
    # plt.close()

    plt.show()
    

# System Latency

In [ ]:
def plot_latency(TASK_RATE, ALGORITHM_LIST):

    df_latency = df_final[df_final['config_rate'] == TASK_RATE]

    sns.set(style="ticks")
    sns.set_palette(color_style)

    ax = plt.subplot()

    ax = sns.barplot(x='config_deadline', y='result_latency', data=df_latency, hue='algorithm', edgecolor='k', estimator=np.mean, errorbar=('ci', 95), capsize=.08, err_kws={'linewidth': 1})

    plt.xlabel(X_LEGEND, fontsize=legend_size)
    plt.ylabel(Y_LEGEND_LATENCY, fontsize=legend_size)

    xticks = np.arange(0,len(DEADLINES),1)

    plt.xticks(xticks, DEADLINES, fontsize=legend_size)
    plt.yticks(fontsize=legend_size)

    # Define os intervalos dos ticks do eixo y para cada subplot
    ax.yaxis.set_major_locator(MultipleLocator(YTICKS_LATENCY))

    plt.ylim(YLIM_LATENCY)

    if WITH_HATCHES == True:
        hatches = HATCHES.values()
        # Loop over the bars
        for bars, hatch in zip(ax.containers, hatches):
            # Set a different hatch for each group of bars
            for bar in bars:
                bar.set_hatch(hatch)

    # Adiciona um bloco de legenda no topo dos subplots com hatches
    legend_handles, names = ax.get_legend_handles_labels()
    final_labels = []
    if WITH_HATCHES == True:
        for alg, handle, hatch in zip(names, legend_handles, HATCHES):
            print(alg, handle, hatch)
            final_name = LABELS[alg]
            final_labels.append(final_name)
            handle.set_hatch(HATCHES[alg])
    else:
        for alg, handle in zip(names, legend_handles):
            final_name = LABELS[alg]
            final_labels.append(final_name)

    ax.legend(
        legend_handles,
        final_labels,
        loc             = 'upper center',
        bbox_to_anchor  = config_legend,
        handletextpad   = 0.2,
        handlelength    = 2.2,
        handleheight    = 1.2,
        columnspacing   = 0.5,
        fancybox        = False,
        frameon         = False,
        ncol            = len(ALGORITHM),
        edgecolor       = 'k',
        fontsize        = font_size_algoritm
    )

    # adiciona grid ao fundo do plot
    plt.grid(color='k', linestyle='--', linewidth=grid_config_lw, axis='both', alpha=grid_config_alpha)

    fig = plt.gcf()
    fig.set_size_inches(x_fig, y_fig)
    fig.savefig(output_path + 'Latencia_' + str(TASK_RATE) + '_' + output_name + formato, dpi=200, bbox_inches = 'tight', pad_inches = 0.05)
    # plt.close()

    plt.show()


# CPU Time

In [ ]:
header = ['cpu_time', 'config_seed', 'config_rate', 'config_deadline', 'algorithm']

df_local = pd.DataFrame(columns=header)

dataframes = []

for algorithm in ALGORITHM:

    for task_rate in RATE:

        for deadline in DEADLINES:

            for seed in SEEDS:
    
                arquivo = path + str(algorithm) + '/SEED_' + str(seed) + '_TIME_radius_2000_resource_1_weight_10_rate_' + str(task_rate) + '_megacycles_' + str(CPU_CYCLE) + '_deadline_' + str(deadline) + '.txt'
                df_local = pd.read_csv(arquivo, sep='\t', names=header)
                df_local['config_seed'] = seed
                df_local['config_rate'] = task_rate
                df_local['config_deadline'] = deadline
                df_local['algorithm'] = algorithm
                dataframes.append(df_local)

                # print(arquivo)

df_final_time = pd.concat(dataframes, ignore_index=True)
df_final_time

In [ ]:
df_final_time['cpu_time_ms'] = df_final_time['cpu_time'] * 1000
df_final_time

In [ ]:
def plot_cputime(TASK_RATE, ALGORITHM_LIST):

    df_time = df_final_time[df_final_time['config_rate'] == TASK_RATE]

    sns.set(style="ticks")
    sns.set_palette(color_style)

    ax = plt.subplot()

    ax = sns.barplot(x='config_deadline', y='cpu_time_ms', data=df_time, hue='algorithm', edgecolor='k', estimator=np.mean, errorbar=('ci', 95), capsize=.08, err_kws={'linewidth': 1})

    plt.xlabel(X_LEGEND, fontsize=legend_size)
    plt.ylabel(Y_LEGEND_CPU_TIME, fontsize=legend_size)

    xticks = np.arange(0,len(DEADLINES),1)

    plt.xticks(xticks, DEADLINES, fontsize=legend_size)
    plt.yticks(fontsize=legend_size)

    plt.ylim(YLIM_CPU_TIME)

    # Define os intervalos dos ticks do eixo y para cada subplot
    ax.yaxis.set_major_locator(MultipleLocator(YTICKS_CPU_TIME))

    # plt.yscale('log')

    if WITH_HATCHES == True:
        hatches = HATCHES.values()
        # Loop over the bars
        for bars, hatch in zip(ax.containers, hatches):
            # Set a different hatch for each group of bars
            for bar in bars:
                bar.set_hatch(hatch)

    # Adiciona um bloco de legenda no topo dos subplots com hatches
    legend_handles, names = ax.get_legend_handles_labels()
    final_labels = []
    if WITH_HATCHES == True:
        for alg, handle, hatch in zip(names, legend_handles, HATCHES):
            print(alg, handle, hatch)
            final_name = LABELS[alg]
            final_labels.append(final_name)
            handle.set_hatch(HATCHES[alg])
    else:
        for alg, handle in zip(names, legend_handles):
            final_name = LABELS[alg]
            final_labels.append(final_name)

    ax.legend(
        legend_handles,
        final_labels,
        loc             = 'upper center',
        bbox_to_anchor  = config_legend,
        handletextpad   = 0.2,
        handlelength    = 2.2,
        handleheight    = 1.2,
        columnspacing   = 0.5,
        fancybox        = False,
        frameon         = False,
        ncol            = len(ALGORITHM),
        edgecolor       = 'k',
        fontsize        = font_size_algoritm
    )

    # adiciona grid ao fundo do plot
    plt.grid(color='k', linestyle='--', linewidth=grid_config_lw, axis='both', alpha=grid_config_alpha)

    fig = plt.gcf()
    fig.set_size_inches(x_fig, y_fig)
    fig.savefig(output_path + 'TempoCPU_' + str(TASK_RATE) + '_' + output_name + formato, dpi=200, bbox_inches = 'tight', pad_inches = 0.05)
    # plt.close()

    plt.show()

# Execução Escalonamento

In [ ]:

RATE = [30,]

for i in RATE:
    plot_scheduled(i, LABELS)
    plot_latency(i, LABELS)
    plot_cost(i, LABELS)
    plot_cputime(i, LABELS)